In [46]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [47]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\cluster.jl:1049


4-element Vector{Int64}:
 22
 23
 24
 25

In [48]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 300
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "TVHAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [49]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [50]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [51]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [52]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 23:	[ Info: Performing boostrap simulation number 2
      From worker 25:	[ Info: Performing boostrap simulation number 4
      From worker 24:	[ Info: Performing boostrap simulation number 3
      From worker 22:	[ Info: Performing boostrap simulation number 1
      From worker 25:	[ Info: Bootstrap 4 generated.
      From worker 22:	[ Info: Bootstrap 1 generated.
      From worker 23:	[ Info: Bootstrap 2 generated.
      From worker 24:	[ Info: Bootstrap 3 generated.


Task (done) @0x0000022b76a87460

In [53]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, 4.407560950664212e-5, -1.4687441339802544e-5, 7.169985843466392e-5, 9.055435825092755e-5, 7.062015244181958e-5, 3.439021106938781e-5, -0.00011397493351859197, -0.00010899187792614709, -4.670433259857958e-5  …  7.053571006824535e-7, 1.2241843002964737e-6, 1.3478917533650114e-6, 1.7572502172823358e-6, 1.834412176692418e-6, 1.8469213725893715e-6, 1.703830483282479e-6, 1.6600019300370977e-6, 1.7126668833504353e-6, 1.919979062893888e-6]
 [NaN, -2.0908133048138e-5, -0.00021800311399584082, 7.192943213335431e-5, -4.5227561847523874e-5, -5.219917560567706e-5, -9.173606649669203e-5, -7.41708265812955e-5, -5.835392167599819e-5, -1.790058270843874e-6  …  -9.341627542126178e-6, -5.1934192152079526e-6, -2.68991733897878e-6, -2.3030025114131777e-6, 8.169057441901261e-7, -4.9973604668784386e-6, -2.057185474235143e-6, -6.661357728921934e-8, 1.6034846336368964e-6, 3.833897157187152e-6]
 [NaN, -0.00012139497119346945, -9.128512471842689e-5, -0.000138479205822474

In [54]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [55]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 6.068519248469274e-6


In [ ]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr